[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/01_wgcna.ipynb )

# R1-Q2 Week 2 — Build per-tissue co-expression networks

This notebook builds two co-expression networks — one for shoot, one for root — using PyWGCNA and the parameters you pre-committed in Week 1.

By the end of this notebook you will have:

- Two per-tissue WGCNA networks built with your pre-committed parameters (`shoot_wgcna.pkl`, `root_wgcna.pkl`).
- Soft-thresholding power diagnostics and module-structure inspection plots for each tissue.
- A `network_summary.json` (per-tissue gene count, module count, soft power used, key parameters) ready as Week 2 EQ#2 input.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

#!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory,
# and load the pre-commit you wrote in Notebook 0.
import json
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

precommit_path = OUTPUT_DIR / "precommit.json"
try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find your pre-commit file.\n"
        f"  Expected at: {precommit_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes `precommit.json` to this location.\n"
    ) from None

print(f"Loaded pre-commit with {len(precommit)} top-level entries.")

## 1. Load the filtered matrices and confirm your pre-committed parameters

In `00_question_orientation.ipynb` you produced two filtered expression matrices — one for shoot samples, one for root — and wrote your network-construction choices to `precommit.json`. This section loads both back into memory and prints your committed choices so they're visible alongside the code that will use them.

This is a deliberate beat, not bookkeeping. The whole point of pre-committing parameters in writing is that you can see them when the analysis runs. If something in the printed summary surprises you, stop and re-read what you wrote in N0 before continuing.

In [ ]:
# Load the filtered shoot and root matrices written by Notebook 0.
import pandas as pd

shoot_path = OUTPUT_DIR / "shoot_filtered.parquet"
root_path  = OUTPUT_DIR / "root_filtered.parquet"

try:
    shoot = pd.read_parquet(shoot_path)
    root  = pd.read_parquet(root_path)
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"\nCould not find a filtered matrix.\n"
        f"  Expected: {shoot_path}\n"
        f"            {root_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes both files to this location.\n"
    ) from None

In [ ]:
# Confirm matrix shapes. Each matrix should be samples (rows) by genes (columns).
print("Shoot matrix:")
print(f"  {shoot.shape[0]} genes  ×  {shoot.shape[1]} samples")
print()
print("Root matrix:")
print(f"  {root.shape[0]} genes  ×  {root.shape[1]} samples")

# After the print:
assert shoot.shape[0] > shoot.shape[1], (
    f"Expected genes × samples (more rows than columns); got {shoot.shape}. "
    "Has N0's on-disk convention changed?"
)
assert root.shape[0] > root.shape[1], (
    f"Expected genes × samples (more rows than columns); got {root.shape}. "
    "Has N0's on-disk convention changed?"
)

Both matrices should have **gene counts in the same ballpark**. N0's MAD filter
used the same percentile cutoff per tissue but applied it independently to each —
exact equality is coincidence, not guarantee. Sample counts depend on which
AtGenExpress samples N0 retained per tissue; in practice these are matched (both
tissues come from the same plants), so equal or near-equal sample counts are
expected. If either shape looks far off from what you expect — for example, an
order-of-magnitude difference between tissues, or sample counts under 50 — go
back to N0 before continuing.

In [ ]:
# Echo the pre-committed network-construction choices you wrote in N0.
nc = precommit["network_construction"]
p  = nc["params"]

print("Pre-committed network-construction choices")
print("-" * 44)
print(f"Method:                {nc['method']}")
print(f"Network type:          {p['networkType']}")
print(f"TOM type:              {p['TOMType']}")
print(f"R² cutoff (scale-free): {p['RsquaredCut']}")
print(f"Minimum module size:   {p['minModuleSize']}")
print(f"Module merge threshold: {p['MEDissThres']}")
print()
print("Reasoning:")
print(nc["reasoning"])

These are the choices you'll feed PyWGCNA in section 2. The soft-thresholding power isn't fixed yet — PyWGCNA will scan a default range of candidate powers and pick the lowest one that achieves scale-free topology at the R² cutoff you pre-committed (0.9). Section 2 will record the chosen power in both `precommit.json` (under `network_construction.applied`) and `network_summary.json` (for Week 2 EQ#2).

With matrices loaded and parameters confirmed, you're ready to build the networks.

## 2. Build the per-tissue co-expression networks

Now you'll build two co-expression networks using PyWGCNA — one for shoot samples, one for root samples — feeding each the parameters you pre-committed.

PyWGCNA's `findModules()` does a lot of work in a single call: it scans candidate soft-thresholding powers, picks the one that achieves your R² cutoff, builds the adjacency matrix and topological overlap matrix, detects modules via dynamic tree cut, and merges modules whose eigengenes are similar enough. Expect each tissue to take a few minutes on Colab; you'll see progress messages as each step runs, plus a diagnostic plot showing the soft-power selection.

The shoot and root networks are built independently. There's no shared state between them.

In [ ]:
# PyWGCNA expects samples in rows, genes in columns.
# Our parquet files store genes in rows (the more common convention upstream),
# so transpose here before handing the matrices to PyWGCNA.
shoot_expr = shoot.T
root_expr  = root.T

print(f"Shoot expression matrix for PyWGCNA: {shoot_expr.shape[0]} samples × {shoot_expr.shape[1]} genes")
print(f"Root expression matrix for PyWGCNA:  {root_expr.shape[0]} samples × {root_expr.shape[1]} genes")

In [ ]:
# Build the shoot network.
import PyWGCNA

shoot_wgcna = PyWGCNA.WGCNA(
    name="shoot",
    species="arabidopsis thaliana",
    geneExp=shoot_expr,
    outputPath=str(OUTPUT_DIR) + "/",
    networkType=p["networkType"],
    TOMType=p["TOMType"],
    RsquaredCut=p["RsquaredCut"],
    minModuleSize=p["minModuleSize"],
    MEDissThres=p["MEDissThres"],
    save=False,  # we save deliberately in S4, not as a side effect here
)

shoot_wgcna.findModules()

In [ ]:
# Build the root network.
root_wgcna = PyWGCNA.WGCNA(
    name="root",
    species="arabidopsis thaliana",
    geneExp=root_expr,
    outputPath=str(OUTPUT_DIR) + "/",
    networkType=p["networkType"],
    TOMType=p["TOMType"],
    RsquaredCut=p["RsquaredCut"],
    minModuleSize=p["minModuleSize"],
    MEDissThres=p["MEDissThres"],
    save=False,
)

root_wgcna.findModules()

Both networks are built. Each WGCNA object now carries the soft-thresholding power that was selected, the adjacency matrix, the topological overlap matrix, the module assignments per gene, and the module eigengenes. The next section records the soft-power that PyWGCNA chose into your pre-commit file and into a fresh network_summary.json for Week 2 EQ#2.

## 3. Record what PyWGCNA did

You pre-committed your network-construction parameters in Notebook 0 — what kind of network, what cutoff, what minimum module size. PyWGCNA just ran the analysis and made some derived choices the pre-commit didn't fix: which soft-thresholding power to use, how many modules to extract. This section records those derived choices in two places.

The first is your pre-commit file (`precommit.json`), under a new `applied` block that sits alongside `params`. The structure becomes: "here's what I said I'd do, and here's what got done." If those two halves ever drift apart, you've made an undocumented choice and the pre-commit's job — making your analytical decisions visible — has failed.

The second is a fresh `network_summary.json` — the small, focused artifact that's part of your Week 2 EQ#2 deliverable. It's a portable summary of what came out of N1.

In [ ]:
# Extract per-tissue results from the WGCNA objects.
def summarize_network(wgcna):
    """Return a dict of the derived choices PyWGCNA made for one tissue."""
    return {
        "soft_thresholding_power": int(wgcna.power),
        "n_samples": int(wgcna.geneExpr.shape[0]),  # PyWGCNA stores samples in dim 0 (obs)
        "n_genes":   int(wgcna.geneExpr.shape[1]),  # genes in dim 1 (var)
        "n_modules_total": int(wgcna.datME.shape[1]),
        "module_names": list(wgcna.getModuleName()),
    }

shoot_summary = summarize_network(shoot_wgcna)
root_summary  = summarize_network(root_wgcna)

print("Shoot:")
for k, v in shoot_summary.items():
    print(f"  {k}: {v}")
print("\nRoot:")
for k, v in root_summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Update precommit.json with the applied block.
precommit["network_construction"]["applied"] = {
    "shoot": shoot_summary,
    "root":  root_summary,
}

with open(precommit_path, "w") as f:
    json.dump(precommit, f, indent=2)

print(f"Updated {precommit_path}")
print(f"  added: network_construction.applied.shoot")
print(f"  added: network_construction.applied.root")

In [ ]:
# Write a fresh network_summary.json for Week 2 EQ#2.
network_summary = {
    "shoot": shoot_summary,
    "root":  root_summary,
    "common_params": {
        "networkType":   precommit["network_construction"]["params"]["networkType"],
        "TOMType":       precommit["network_construction"]["params"]["TOMType"],
        "RsquaredCut":   precommit["network_construction"]["params"]["RsquaredCut"],
        "minModuleSize": precommit["network_construction"]["params"]["minModuleSize"],
        "MEDissThres":   precommit["network_construction"]["params"]["MEDissThres"],
    },
}

network_summary_path = OUTPUT_DIR / "network_summary.json"
with open(network_summary_path, "w") as f:
    json.dump(network_summary, f, indent=2)

print(f"Wrote {network_summary_path}")

A few things to notice in the numbers above.

For shoot, PyWGCNA selected soft-thresholding power **11**. The scale-free fit
climbed smoothly from R² ≈ 0.02 at power 1, crossed your pre-committed R² ≥ 0.9
cutoff at power 11 (R² = 0.908), and continued climbing toward 0.95 at higher
powers. Mean connectivity at the selected power dropped to ~1.5 — well inside
the textbook-healthy range. The selection rule worked as documented: PyWGCNA
picked the first power to clear the cutoff.

For root, PyWGCNA selected power **6**. Root's scale-free fit reached the 0.9
cutoff much faster than shoot's — R² was already 0.77 at power 2 and 0.91 by
power 6. The curve then plateaus through power 19 rather than continuing to
climb. Different powers for different tissues is the expected outcome of
building separate per-tissue networks (it's why R1-Q2 does so in the first
place); the difference reflects real structural differences between root and
shoot expression networks.

Both tissues produced **9 modules** after dynamic tree cut and eigengene-distance
merging. The shared count is coincidental — shoot's path was 11 → 10 → 9 over
two merge rounds, root's was 14 → 12 → 11 → 9 over three. The module names
differ substantially between tissues (only `black`, `darkgrey`, `dimgrey`,
`gainsboro`, and `lightcoral` appear in both), which is what we'd expect when
the same algorithm finds biologically distinct co-expression structure in two
different tissues.

The `applied` block in `precommit.json` records what PyWGCNA actually did for
each tissue alongside the parameters you pre-committed. A reader of the artifact
can see at a glance which decisions PyWGCNA made (soft power, module count) and
which you locked in advance (network type, TOM type, R² cutoff, minimum module
size, merge threshold).